In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

import re
import random

# Обработка датасета

In [2]:
ds = load_dataset("t-tech/T-math", split="train")

In [3]:
ANSWER_TAG = '[ANSWER]'
# дробь может быть в виде обычной дроби через /
NUM_RE = re.compile(
    r"[-+]?(?:\d+\s*/\s*\d+|(?:\d+|\d+)(?:[.,]\d+)?)"
)

def extract_answer(text: str):
    if text is None:
        return False, None
    # смотрим только на ответ модели
    assistant_text = text.split('assistant')[-1]
    if ANSWER_TAG not in assistant_text:
        return False, None
    answer = assistant_text.split(ANSWER_TAG)[-1] # могут быть ранние упоминания в промпте
    return True, answer

def normalize(s: str) -> str:
    if s is None:
        return ""
    raw = str(s).strip().lower()
    m = NUM_RE.search(raw)
    if m:
        num = m.group(0).replace(" ", "").replace(",", ".")
        try:
            if "." in num:
                v = str(float(num)).rstrip("0").rstrip(".")
            else:
                v = str(int(num))
            return v
        except ValueError:
            return num
    raw = raw
    raw = re.sub(r"\\s+", " ", raw).rstrip(" .,")
    return raw

def compute_metrics(preds, refs):
    parsed = 0
    correct = 0
    for p, r in zip(preds, refs):
        ok, val = extract_answer(p)
        if ok:
            parsed += 1
            if normalize(val) == r:
                correct += 1
    n = len(refs)
    return {
        "format_rate": parsed / n if n else 0.0,
        "accuracy": correct / n if n else 0.0,
        "count": n,
    }

preds = ["[SOLUTION] 2+2=4\n[ANSWER] 4.0", "ответ без формата", "[ANSWER] число"]
refs = ["4", "что-то", "0"]
print(compute_metrics(preds, refs))

{'format_rate': 0.6666666666666666, 'accuracy': 0.3333333333333333, 'count': 3}


# Baseline & few-shot

In [4]:
%env HF_ENDPOINT=https://hf-mirror.com

env: HF_ENDPOINT=https://hf-mirror.com


In [4]:
MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, 
                                             device_map="auto")
ds = load_dataset("t-tech/T-math", split="train")

def generate_baseline(question: str) -> str:
    messages = [{
        "role": "user",
        "content": (
            "Реши задачу и выведи ответ ТОЛЬКО число после '[ANSWER]'.\n"
            f"Задача: {question}\n"
        ),
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors='pt'
    )
    with torch.no_grad():
        out = model.generate(input_ids=inputs.to(model.device), 
                             max_new_tokens=64,
                             do_sample=False)
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

preds_raw = [generate_baseline(r["question"]) for r in tqdm(ds)]
refs = [r["verifiable_answer"] for r in ds]
print("Пример предсказания:\n", preds_raw[0])

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  0%|          | 0/331 [00:00<?, ?it/s]

AttributeError: 

In [11]:
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Загружаем датасет
ds = load_dataset("t-tech/T-math", split="train")

def generate_few_shot(question: str, idx: int, n_examples: int = 3) -> str:
    """
    question: текущий вопрос
    idx: индекс вопроса в датасете
    n_examples: сколько few-shot примеров использовать
    """
    # Выбираем случайные примеры, исключая текущий
    candidates = list(range(len(ds)))
    candidates.remove(idx)
    sample_idxs = random.sample(candidates, n_examples)

    # Формируем диалог few-shot
    messages = [
        {
            "role": "system",
            "content": "Реши задачу и выведи ответ ТОЛЬКО числом после '[ANSWER]'."
        }
    ]

    for ex_idx in sample_idxs:
        ex = ds[ex_idx]
        messages.append({
            "role": "user",
            "content": f"Задача: {ex['question']}"
        })
        messages.append({
            "role": "assistant",
            "content": f"[ANSWER] {ex['verifiable_answer']}"
        })

    # Добавляем текущую задачу
    messages.append({
        "role": "user",
        "content": f"Задача: {question}"
    })

    # Токенизация и генерация
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors='pt'
    )
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs.to(model.device),
            max_new_tokens=64,
            do_sample=False
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

preds_raw = [
    generate_few_shot(r["question"], i, n_examples=3)
    for i, r in enumerate(tqdm(ds))
]
refs = [r["verifiable_answer"] for r in ds]

print("Пример предсказания:\n", preds_raw[0])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  0%|          | 0/331 [00:00<?, ?it/s]

AttributeError: 

# Batch generation (CoT)

In [9]:
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto"
).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_dataset("t-tech/T-math", split="train")

def build_messages(question: str, idx: int, n_examples: int = 3):
    """
    Собирает чат-сообщения для few-shot как в исходной версии.
    """
    candidates = [i for i in range(len(ds)) if len(ds[i]['solutions']) > 0 and i != idx]
    sample_idxs = random.sample(candidates, n_examples)

    messages = [
        {
            "role": "system",
            "content": "Напиши '[SOLUTION]' и после него решение задачи последовательно по шагам. "
                       "Напиши '[ANSWER]' и выведи ответ ТОЛЬКО число.\n"
        }
    ]
    for ex_idx in sample_idxs:
        ex = ds[ex_idx]
        messages.append({
            "role": "user",
            "content": f"Задача: {ex['question']}"
        })
        messages.append({
            "role": "assistant",
            "content": f"[SOLUTION] {ex['solutions'][0]} [ANSWER] {ex['verifiable_answer']}"
        })
    messages.append({
        "role": "user",
        "content": f"Задача: {question}"
    })
    return messages

def generate_batch(batch_indices, n_examples: int = 3):
    """
    Батчевая генерация по списку индексов в датасете.
    Возвращает список декодированных ответов (как и раньше, с промптом).
    """
    batch_messages = [
        build_messages(ds[i]["question"], i, n_examples) for i in batch_indices
    ]

    chat_inputs = tokenizer.apply_chat_template(
        batch_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(chat_inputs, 
                       padding=True,
                       return_tensors='pt')

    with torch.no_grad():
        out = model.generate(
            **inputs.to(model.device),
            max_new_tokens=1024,
            do_sample=True
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)

BATCH_SIZE = 4
preds_raw = []

for start in tqdm(range(0, len(ds), BATCH_SIZE)):
    batch_idxs = list(range(start, min(start + BATCH_SIZE, len(ds))))
    preds_raw.extend(generate_batch(batch_idxs, n_examples=3))

refs = [r["verifiable_answer"] for r in ds]

print("Пример предсказания:\n", preds_raw[0])

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  0%|          | 0/83 [00:00<?, ?it/s]

Пример предсказания:
 system
Напиши '[SOLUTION]' и после него решение задачи последовательно по шагам. Напиши '[ANSWER]' и выведи ответ ТОЛЬКО число.

user
Задача: Дан треугольник ABC. Пусть точка I — центр его вписанной окружности, а точки P и Q — середины сторон AB и AC соответственно. Оказалось, что ∠PIQ + ∠BIC = 180°. Найдите длину отрезка BC, если AB = 20 и AC = 14.
assistant
[SOLUTION] Из условия следует, что ∠BIP + ∠CIQ = 180°. Кроме того, отметим, что PQ || BC как средняя линия треугольника.

Проведём к вписанной окружности треугольника \(ABC\) касательную, параллельную отрезку \(BC\). Обозначим через \(P'\) и \(Q'\) точки пересечения этой касательной со сторонами \(AB\) и \(AC\) соответственно. Поскольку в трапецию \(P'Q'CB\) вписана окружность с центром \(I\), то точка \(I\) является точкой пересечения биссектрис всех четырёх углов этой трапеции. Так как \(P'Q' \parallel BC\), то \(\angle P'Q'C + \angle BCQ' = 180^\circ\). Тогда \(\angle IQ'C + \angle ICQ' = 90^\circ\), откуд

# Reasoning